In [ ]:
import os

os.chdir("..")

In [ ]:
from fsdc_calories import DataCal

import polars as pl
import pandas as pd

dc = DataCal()

In [ ]:
df = pl.read_csv("data/external/hts-calorie-pyramid.csv")
# df = df.filter(pl.col("Rank") != "TOTAL")
df

In [ ]:
df2 = dc.gen_nuti_data().filter(pl.col("fiscal_year") == 2025).select("total_calories", "hts_code")
df = df.join(df2, left_on="HTS", right_on="hts_code", validate="1:1")
df.with_columns(
    test=pl.col("Net Import Calories") / pl.col("total_calories")
)

In [ ]:
df = pl.from_pandas(pd.read_stata("data/external/datos completos con arroz 23 de julio de 2026.dta"))
# df.filter((pl.col("schedule_b") == 401) & (pl.col("fiscal_year") == 2025))
df

In [ ]:
data = dc.process_int_jp(time_frame="fiscal", level="hts")
# data = data.with_columns(
#     net_qty=pl.when(pl.col("net_qty") < 0)
#     .then(pl.lit(0))
#     .otherwise(pl.col("net_qty"))
# )

# 3. Match HTS lengths (slicing down to 4 digits to match nutri matrix)
data = data.with_columns(hts_code=pl.col("hts_code").str.slice(0, 4))
data = data.group_by(["hts_code", "fiscal_year"]).agg(pl.col("net_qty", "imports_qty", "exports_qty").sum())
data.filter(pl.col("hts_code") == "0401").filter(pl.col("fiscal_year") == 2025)

In [ ]:
df = pl.read_parquet("src/fsdc_calories/data/local_nutri_data.parquet")#.filter((pl.col("hts") == "0401") & (pl.col("fiscal_year") == 2025))
df.group_by(["fiscal_year"]).agg(pl.col("produccion_kg").sum()).sort("fiscal_year")

In [ ]:
pl.read_parquet("src/fsdc_calories/data/nutri_matrix.parquet").filter(pl.col("schedule_b") == "0401")

In [ ]:
data = dc.process_int_jp(time_frame="fiscal", level="hts")
data.group_by(["fiscal_year"]).agg(pl.col("*").exclude("hts_code").sum()).sort("fiscal_year")

In [ ]:
1.9640e10	

In [ ]:
19_640_000_000.0